Title

Import

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from dataclasses import dataclass
import torch

import rainflow
from scipy.optimize import minimize, differential_evolution, basinhopping
from scipy.optimize import LinearConstraint, NonlinearConstraint
import warnings
import pyomo.environ as pyo

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pandas as pd
from typing import Tuple

data and aparameters

In [ ]:
@dataclass
class BatteryParams:
    
    T_horizon: int = 24
    T_min: int = 15
    delta_t: float = T_min/60  # Time step in hours (15 min = 0.25 hr, 1 hr = 1.0)
    T: int = int(T_horizon / delta_t)  # Number of time steps
    
    # Battery specifications
    battery_capacity: float = 10.0  # kWh
    max_charge_power: float = 4.8  # kW
    max_discharge_power: float = 4.8  # kW
    charge_efficiency: float = 0.92
    discharge_efficiency: float = 0.92
    
    # SOC constraints (as fraction of capacity: 0.1 = 10%, 0.9 = 90%)
    soc_min: float = 0.1  # 10% minimum (1 kWh for 10 kWh battery)
    soc_max: float = 0.9  # 90% maximum (9 kWh for 10 kWh battery)
    initial_soc: float = 0.5  # 50%
    final_soc: float = 0.5   # 50%
    
    # Economic parameters
    feed_in_tariff_ratio: float = 0.3
    battery_replacement_cost: float = 300  # $
    
    # Rainflow degradation model
    cycle_a: float = 5.24e-4
    cycle_b: float = 2.03
    
    # PWL discretization
    J: int = 16
    
    # L2O-specific training parameters
    solar_capacity: float = 16.0
    soc_penalty_weight: float = 20.0
    terminal_penalty_weight: float = 10.0
    track_penalty_weight: float = 0.0
    temperature_init: float = 5.0
    temperature_min: float = 0.1
    temperature_decay: float = 0.99
    degradation_loss_weight: float = 1.0
    
    lambda_rainflow: float = 0.5
    lambda_proxy: float = 0.5

    def get_xu_degradation_costs(self):      
        J = self.J
        R = self.battery_replacement_cost * self.battery_capacity
    
        def Phi(delta):
            return self.cycle_a * delta**self.cycle_b
    
        C_dis_kWh = np.zeros(J)
        for j in range(J):
            delta_j = (j + 1) / J
            delta_j_minus_1 = j / J
            C_dis_kWh[j] = R * J / (self.discharge_efficiency * self.battery_capacity) * (Phi(delta_j) - Phi(delta_j_minus_1))
    
        return C_dis_kWh


def generate_training_data_netmeter_normal(num_samples, params: BatteryParams,feed_in_ratio=0.3):

    T = params.T
    delta_t = params.delta_t
    time_hours = np.arange(T) * delta_t  # Convert timestep index to hours (0, 0.25, 0.5, ..., 23.75)
    solar_profiles = np.zeros((num_samples, T))
    demand_profiles = np.zeros((num_samples, T))
    price_buy_profiles = np.zeros((num_samples, T))
    price_sell_profiles = np.zeros((num_samples, T))
    
    for i in range(num_samples):
        # Solar generation: peaks at midday
        solar_peak = np.random.uniform(3.0, 5.0)
        solar = solar_peak * np.maximum(0, np.sin(np.pi * (time_hours - 6) / 12))
        solar = solar + np.random.normal(0, 0.2, T)
        solar_profiles[i] = np.maximum(0, solar)
        
        # Demand: morning and evening peaks
        morning_peak = np.random.uniform(2.0, 3.0)
        evening_peak = np.random.uniform(3.0, 4.5)
        demand = morning_peak * np.exp(-((time_hours - 7)**2) / 8) + \
                 evening_peak * np.exp(-((time_hours - 19)**2) / 8) + \
                 np.random.uniform(0.5, 1.0)
        demand = demand + np.random.normal(0, 0.1, T)
        demand_profiles[i] = np.maximum(0, demand)
        
        # Time-of-use pricing
        base_price = np.random.uniform(0.10, 0.15)
        peak_price = np.random.uniform(0.25, 0.35)
        is_peak = (time_hours >= 16) & (time_hours <= 21)
        price_buy = np.where(is_peak, peak_price, base_price)
        price_buy = price_buy + np.random.normal(0, 0.02, T)
        price_buy_profiles[i] = np.maximum(0.05, price_buy)
        
        # Sell price: reduced from buy price, further reduced at midday
        price_sell = price_buy * feed_in_ratio
        midday_factor = np.exp(-((time_hours - 12)**2) / 16)
        price_sell = price_sell * (1 - 0.3 * midday_factor)
        price_sell_profiles[i] = np.maximum(0.02, price_sell)
    
    return (torch.from_numpy(solar_profiles).float(),
            torch.from_numpy(demand_profiles).float(),
            torch.from_numpy(price_buy_profiles).float(),
            torch.from_numpy(price_sell_profiles).float())
    


# Create parameter instances
params = BatteryParams(J=16)
params_pwl = BatteryParams(J=16)
params_minlp = BatteryParams()
params_l2o = BatteryParams(J=16)



In [ ]:
def xu_battery_optimization(params, solar, demand, price_buy, price_sell=None):

    T = params.T
    J = params.J
    Ecap = params.battery_capacity
    Pmax = params.max_charge_power
    eta = params.charge_efficiency
    soc_min = params.soc_min
    soc_max = params.soc_max
    soc_initial = params.initial_soc
    soc_final = params.final_soc
    dt = params.delta_t
    
    # Get degradation costs
    C_dis_kWh = params.get_xu_degradation_costs()
    
    solar = np.asarray(solar).reshape(T)
    demand = np.asarray(demand).reshape(T)
    price_buy = np.asarray(price_buy).reshape(T)
    if price_sell is None:
        price_sell = params.feed_in_tariff_ratio * price_buy
    else:
        price_sell = np.asarray(price_sell).reshape(T)
    
    soc_seg_max = Ecap / J  
    
    # Initial segments: distribute energy from 0% to soc_initial
    soc0_absolute = soc_initial * Ecap 
    soc0_segments = np.zeros(J)
    remaining = soc0_absolute
    for j in range(J):  # Fill from bottom (j=0 is deepest)
        amount = min(soc_seg_max, remaining)
        soc0_segments[j] = amount
        remaining -= amount
    
    # Decision variables
    ch = cp.Variable((T, J), nonneg=True)
    dis = cp.Variable((T, J), nonneg=True)
    soc = cp.Variable((T, J))
    p_ch = cp.Variable(T, nonneg=True)
    p_dis = cp.Variable(T, nonneg=True)
    p_import = cp.Variable(T, nonneg=True)
    p_export = cp.Variable(T, nonneg=True)
    v = cp.Variable(T, boolean=True)
    
    constraints = []
    
    # Power aggregation constraints
    for t in range(T):
        constraints += [
            p_ch[t] == cp.sum(ch[t, :]),
            p_dis[t] == cp.sum(dis[t, :])
        ]
    
    # Power limits and mode constraints
    for t in range(T):
        constraints += [
            p_ch[t] <= Pmax * (1 - v[t]),
            p_dis[t] <= Pmax * v[t]
        ]
    
    # SOC dynamics (segments represent energy from bottom of battery)
    for t in range(T):
        for j in range(J):
            if t == 0:
                constraints += [
                    soc[t, j] == soc0_segments[j] + (ch[t, j] * eta - dis[t, j] / eta) * dt
                ]
            else:
                constraints += [
                    soc[t, j] == soc[t-1, j] + (ch[t, j] * eta - dis[t, j] / eta) * dt
                ]
            # Each segment bounded by segment capacity 
            constraints += [
                soc[t, j] >= 0,
                soc[t, j] <= soc_seg_max
            ]
    

    # SOC bounds based on full capacity

    for t in range(T):
        total_energy = cp.sum(soc[t, :])  # Total energy in battery
        # Enforce 10%-90% bounds
        constraints += [
            total_energy >= soc_min * Ecap,  # At least 1 kWh (10%)
            total_energy <= soc_max * Ecap   # At most 9 kWh (90%)
        ]
    

    # Final SOC constraint

    constraints += [
        cp.sum(soc[T-1, :]) == soc_final * Ecap
    ]
    
    # Energy balance
    for t in range(T):
        constraints += [
            p_import[t] - p_export[t] == demand[t] - solar[t] + p_ch[t] - p_dis[t]
        ]
    
    # Objective function
    import_cost = cp.sum(cp.multiply(price_buy, p_import)) * dt
    export_revenue = cp.sum(cp.multiply(price_sell, p_export)) * dt
    
    # Degradation cost 
    degradation_cost = 0
    for t in range(T):
        degradation_cost += cp.sum(cp.multiply(C_dis_kWh, dis[t, :])) * dt
    
    obj = cp.Minimize(import_cost - export_revenue + degradation_cost)
    
    problem = cp.Problem(obj, constraints)
    
    variables = {
        'ch': ch,
        'dis': dis,
        'soc': soc,
        'p_ch': p_ch,
        'p_dis': p_dis,
        'p_import': p_import,
        'p_export': p_export,
        'v': v,
        'C_dis_kWh': C_dis_kWh
    }
    
    return problem, variables

def solve_xu_optimization(problem, variables, solar, demand, price_buy, price_sell, params, verbose=True):

    T = variables['p_ch'].shape[0]
    dt = params.delta_t 
    
    # Try solvers
    solvers_to_try = [
        (cp.GUROBI, "Gurobi"),
        (cp.GLPK_MI, "GLPK_MI"),
        (cp.CBC, "CBC"),
    ]
    
    for solver, name in solvers_to_try:
        if solver in cp.installed_solvers():
            try:
                print(f"🔄 Trying solver: {name}...")
                problem.solve(solver=solver, verbose=verbose)
                
                if problem.status in ['optimal', 'optimal_inaccurate']:
                    print(f"✅ {name} solved successfully!")
                    break
                else:
                    print(f"⚠️ {name} returned status: {problem.status}")
                    continue
                    
            except Exception as e:
                print(f"❌ {name} failed: {str(e)[:100]}")
                continue
    
    if problem.status not in ['optimal', 'optimal_inaccurate']:
        print(f"\n❌ All solvers failed. Final status: {problem.status}")
        return {
            'status': problem.status,
            'feasible': False
        }
    
    # Absolute SOC in kWh
    total_soc = np.sum(variables['soc'].value, axis=1)  
    
    # Battery power 
    net_power = variables['p_ch'].value - variables['p_dis'].value
    
    C_dis_kWh = variables['C_dis_kWh']
    
    # Calculate degradation cost (discharge energy × cost per kWh)
    degr_cost = 0
    for t in range(T):
        degr_cost += np.sum(C_dis_kWh * variables['dis'].value[t, :]) * dt
    
    # COST CALCULATIONS - power × price × time
    p_import_power = variables['p_import'].value  # kW
    p_export_power = variables['p_export'].value  # kW
    
    import_cost = np.sum(price_buy * p_import_power * dt)  # $/kWh × kW × h = $
    export_revenue = np.sum(price_sell * p_export_power * dt)  # $/kWh × kW × h = $
    
    results = {
        'status': problem.status,
        'feasible': True,
        'total_cost': problem.value,
        'import_cost': import_cost,
        'export_revenue': export_revenue,
        'degradation_cost': degr_cost,
        'net_power': net_power,  # Added for consistency
        'p_ch': variables['p_ch'].value,
        'p_dis': variables['p_dis'].value,
        'p_import': p_import_power,
        'p_export': p_export_power,
        'import_power': p_import_power,  # Alias
        'export_power': p_export_power,  # Alias
        'soc': total_soc,  # Absolute SOC in kWh (not fractional!)
        'soc_segments': variables['soc'].value,
        'ch_segments': variables['ch'].value,
        'dis_segments': variables['dis'].value,
        'v': variables['v'].value,
        'C_dis_kWh': C_dis_kWh
    }
    
    print("\n✅ Optimization successful!")
    print(f"Total cost: ${results['total_cost']:.2f}")
    print(f"  - Import cost: ${results['import_cost']:.2f}")
    print(f"  - Export revenue: ${results['export_revenue']:.2f}")
    print(f"  - Degradation cost: ${results['degradation_cost']:.2f}")
    print(f"  - SOC range: [{total_soc.min():.2f}, {total_soc.max():.2f}] kWh")
    print(f"  - SOC range (%): [{total_soc.min()/params.battery_capacity*100:.1f}%, {total_soc.max()/params.battery_capacity*100:.1f}%]")
    
    return results

In [ ]:
def rainflow_degradation_exact(soc_trajectory, params):

    Ecap = params.battery_capacity
    Crep = params.battery_replacement_cost * Ecap
    a = params.cycle_a if hasattr(params, 'cycle_a') else 5.24e-4
    b = params.cycle_b if hasattr(params, 'cycle_b') else 2.03
    
    # Normalize to fraction (0-1 range)
    soc_normalized = soc_trajectory / Ecap
    
    # Extract rainflow cycles
    cycles = list(rainflow.extract_cycles(soc_normalized))
    
    # Accumulate degradation stress
    total_stress = 0
    for c in cycles:
        rng, mean, count, i_start, i_end = c
        
        soc_start = soc_normalized[i_start]
        soc_end = soc_normalized[i_end]
        
        #  Only count full cycles and discharge half-cycles
        if count == 1.0:  # Full cycle
            total_stress += a * (rng ** b)
        elif count == 0.5 and soc_start > soc_end:  # Discharge half-cycle only
            total_stress += a * (rng ** b)
    
    degradation_cost = Crep * total_stress
    
    return degradation_cost, cycles


def compute_rainflow_gradient(soc_trajectory, params, delta=1e-4):
    
    T = len(soc_trajectory)
    gradient = np.zeros(T)
    
    base_cost, _ = rainflow_degradation_exact(soc_trajectory, params)
    
    for t in range(T):
        soc_perturbed = soc_trajectory.copy()
        soc_perturbed[t] += delta
        
        perturbed_cost, _ = rainflow_degradation_exact(soc_perturbed, params)
        gradient[t] = (perturbed_cost - base_cost) / delta
    
    return gradient


def segments_to_absolute_soc(soc_segments, params):

    if soc_segments.ndim == 2:
        soc_absolute = np.sum(soc_segments, axis=1)
    else:
        soc_absolute = soc_segments
    
    return soc_absolute


def absolute_soc_to_segments(soc_absolute, params):
    T = len(soc_absolute)
    J = params.J
    Ecap = params.battery_capacity
    
    # Each segment represents energy based on FULL capacity
    soc_seg_max = Ecap / J
    
    soc_segments = np.zeros((T, J))
    
    for t in range(T):
        # Total energy in battery at time t
        total_energy = soc_absolute[t]
        total_energy = np.clip(total_energy, 0, Ecap)
        
        # Distribute across segments from bottom (j=0) to top (j=J-1)
        remaining = total_energy
        for j in range(J):
            amount = min(soc_seg_max, remaining)
            soc_segments[t, j] = amount
            remaining -= amount
    
    return soc_segments

In [ ]:
class RainflowFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, soc_trajectory, params):
        batch_size = soc_trajectory.shape[0]
        T = soc_trajectory.shape[1]
        cycle_costs = torch.zeros(batch_size, device=soc_trajectory.device)
    
        all_peak_indices = []
        all_valley_indices = []
        all_cycle_depths = []
        all_cycle_counts = []  # NEW: Store cycle counts
    
        soc_np = soc_trajectory.detach().cpu().numpy()
        R = params.battery_replacement_cost * params.battery_capacity
    
        for i in range(batch_size):
            soc_normalized = soc_np[i] / params.battery_capacity
            cycles = list(rainflow.extract_cycles(soc_normalized))
        
            sample_peaks = []
            sample_valleys = []
            sample_depths = []
            sample_counts = []
        
            cost = 0.0
            for cycle in cycles:
                cycle_range = cycle[0]
                cycle_count = cycle[2]
                start_idx = cycle[3]
                end_idx = cycle[4]
                
                soc_start = soc_normalized[start_idx]
                soc_end = soc_normalized[end_idx]
            
                # ✅ FIXED: Include ALL cycles (both full and half)
                # Determine peak and valley
                if cycle_count == 1.0:  # Full cycle
                    include_cycle = True
                elif cycle_count == 0.5:  # Half cycle
                    include_cycle = (soc_start > soc_end)  # Only discharging
                else:
                    include_cycle = False
                
                if include_cycle:
                    # Determine peak and valley
                    if soc_start > soc_end:  # Discharge
                        peak_idx = start_idx
                        valley_idx = end_idx
                    else:  # Charge (only for full cycles)
                        peak_idx = end_idx
                        valley_idx = start_idx
    
                    sample_peaks.append(peak_idx)
                    sample_valleys.append(valley_idx)
                    sample_depths.append(cycle_range)
                    sample_counts.append(cycle_count)  # Keep for backward pass

                    cost += params.cycle_a * (cycle_range ** params.cycle_b) * R
        
        
            cycle_costs[i] = float(cost)
            all_peak_indices.append(sample_peaks)
            all_valley_indices.append(sample_valleys)
            all_cycle_depths.append(sample_depths)
            all_cycle_counts.append(sample_counts)
    
        ctx.save_for_backward(soc_trajectory)
        ctx.params = params
        ctx.peak_indices = all_peak_indices
        ctx.valley_indices = all_valley_indices
        ctx.cycle_depths = all_cycle_depths
        ctx.cycle_counts = all_cycle_counts  # NEW
    
        return cycle_costs
   
    
    @staticmethod
    def backward(ctx, grad_output):
        soc_trajectory, = ctx.saved_tensors
        params = ctx.params
        
        batch_size = soc_trajectory.shape[0]
        T = soc_trajectory.shape[1]
        device = soc_trajectory.device
        
        lambda_rainflow = params.lambda_rainflow
        lambda_proxy = params.lambda_proxy
        
        R = params.battery_replacement_cost * params.battery_capacity
        
        # ===== Part 1: Exact Rainflow Gradient (Sparse) =====
        grad_rainflow = torch.zeros_like(soc_trajectory)
        
        for i in range(batch_size):
            peaks = ctx.peak_indices[i]
            valleys = ctx.valley_indices[i]
            depths = ctx.cycle_depths[i]
            counts = ctx.cycle_counts[i]  # NEW
            
            # Multiply gradient by cycle_count
            for peak_idx, valley_idx, depth, count in zip(peaks, valleys, depths, counts):
                # Gradient magnitude includes cycle_count
                grad_magnitude = params.cycle_a * params.cycle_b * (depth ** (params.cycle_b - 1)) * R
                grad_magnitude = grad_magnitude / params.battery_capacity
    
                grad_rainflow[i, peak_idx] += grad_magnitude
                grad_rainflow[i, valley_idx] -= grad_magnitude
        
        # ===== Part 2: Proxy Gradient (Dense) =====
        with torch.enable_grad():
            soc_temp = soc_trajectory.detach().requires_grad_(True)
            delta = soc_temp[:, 1:] - soc_temp[:, :-1]
            delta_normalized = delta / params.battery_capacity
            
            # Proxy
            proxy_cost_per_sample = (params.cycle_a * (torch.abs(delta_normalized) + 1e-6) ** params.cycle_b).sum(dim=1)
            proxy_cost_per_sample = proxy_cost_per_sample * R
            
            weighted_proxy = (proxy_cost_per_sample * grad_output).sum()
            grad_proxy = torch.autograd.grad(weighted_proxy, soc_temp)[0]
        
        # ===== Combine Gradients =====
        grad_total = lambda_rainflow * grad_rainflow + lambda_proxy * grad_proxy
        grad_total = grad_total * grad_output.view(-1, 1)
        grad_total = torch.clamp(grad_total, -1.0, 1.0)
        
        return grad_total, None


class L2O(nn.Module):
    def __init__(self, params: BatteryParams, hidden_dim=1024, 
                 degradation_weight_start=4.0, degradation_weight_end=2.5):
        super().__init__()
        self.params = params
        self.T = params.T
        self.temperature = params.temperature_init
        self.degradation_weight_start = degradation_weight_start
        self.degradation_weight_end = degradation_weight_end
        self.current_degradation_weight = degradation_weight_start

        input_dim = 4 * self.T

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.mode_head = nn.Linear(hidden_dim, self.T * 3)
        self.power_head = nn.Linear(hidden_dim, self.T * 2)

        with torch.no_grad():
            self.mode_head.bias.data = self.mode_head.bias.data.reshape(self.T, 3)
            self.mode_head.bias.data[:, 0] = 0.2
            self.mode_head.bias.data[:, 1] = 0.2
            self.mode_head.bias.data[:, 2] = -0.4
            self.mode_head.bias.data = self.mode_head.bias.data.reshape(-1)

    def forward(self, solar, demand, price_buy, price_sell):
        batch_size = solar.shape[0]

        price_spread = price_buy - price_sell       
        demand_max = 35.0 
        
        inputs = torch.cat([
            solar / self.params.solar_capacity,
            demand / demand_max,  
            price_buy / (price_buy.max(dim=1, keepdim=True)[0] + 1e-6),
            price_spread / (price_spread.max(dim=1, keepdim=True)[0] + 1e-6)
        ], dim=1)

        hidden = self.encoder(inputs)
        mode_logits = self.mode_head(hidden).reshape(batch_size, self.T, 3)

        if self.training:
            mode = F.gumbel_softmax(mode_logits, tau=self.temperature, hard=True)
        else:
            mode = F.one_hot(mode_logits.argmax(dim=-1), num_classes=3).float()

        powers = self.power_head(hidden).reshape(batch_size, self.T, 2)
        power_charge_raw = torch.sigmoid(powers[..., 0]) * self.params.max_charge_power
        power_discharge_raw = torch.sigmoid(powers[..., 1]) * self.params.max_discharge_power

        power_charge = mode[..., 0] * power_charge_raw
        power_discharge = mode[..., 1] * power_discharge_raw

        soc = self.compute_soc(power_charge, power_discharge)
        net_power = demand - solar + power_charge - power_discharge

        return {
            'power_charge': power_charge,
            'power_discharge': power_discharge,
            'mode': mode,
            'soc': soc,
            'net_power': net_power
        }

    def compute_soc(self, power_charge, power_discharge):
        batch_size = power_charge.shape[0]
        device = power_charge.device

        soc_init = self.params.initial_soc * self.params.battery_capacity
        soc = torch.full((batch_size,), soc_init, device=device)
        soc_trajectory = [soc.clone()]

        dt = self.params.delta_t

        for t in range(self.T):
            soc_change = (power_charge[:, t] * self.params.charge_efficiency -
                          power_discharge[:, t] / self.params.discharge_efficiency) * dt
            soc = soc + soc_change
            soc_trajectory.append(soc.clone())

        return torch.stack(soc_trajectory, dim=1)

    def compute_loss(self, solar, demand, price_buy, price_sell):
        output = self.forward(solar, demand, price_buy, price_sell)
        
        dt = self.params.delta_t
        
        net_power = output['net_power']
        import_power = F.relu(net_power)
        export_power = F.relu(-net_power)
        import_cost = (import_power * price_buy * dt).sum(dim=1)
        export_revenue = (export_power * price_sell * dt).sum(dim=1)
        economic_cost = import_cost - export_revenue

        soc_full = output['soc']
        soc_min_abs = self.params.soc_min * self.params.battery_capacity
        soc_max_abs = self.params.soc_max * self.params.battery_capacity
        lower_viol = F.relu(soc_min_abs - soc_full)
        upper_viol = F.relu(soc_full - soc_max_abs)
        soc_violations = (lower_viol + upper_viol).mean(dim=1)

        final_soc = output['soc'][:, -1]
        target_soc = self.params.final_soc * self.params.battery_capacity
        terminal_penalty = F.smooth_l1_loss(final_soc, torch.full_like(final_soc, target_soc), reduction='none')

        computed_net = demand - solar + output['power_charge'] - output['power_discharge']
        track_residual = (net_power - computed_net).pow(2).mean(dim=1)

        cycling_cost = RainflowFunction.apply(output['soc'], self.params)

        total_loss = (
            economic_cost.mean() +
            self.current_degradation_weight * cycling_cost.mean() +
            self.params.soc_penalty_weight * soc_violations.mean() +
            self.params.terminal_penalty_weight * terminal_penalty.mean() +
            self.params.track_penalty_weight * track_residual.mean()
        )

        return total_loss

    def update_curriculum(self, epoch, total_epochs):
        progress = epoch / total_epochs
        smooth_progress = progress ** 0.5
        
        self.current_degradation_weight = (
            self.degradation_weight_start * (1 - smooth_progress) + 
            self.degradation_weight_end * smooth_progress
        )
        
        self.temperature = (
            self.params.temperature_init * (1 - progress) + 
            self.params.temperature_min * progress
        )
        
        return self.current_degradation_weight, self.temperature

    
def train_l2o_model(model, train_data, n_epochs=1500, batch_size=48):  # More epochs, smaller batch
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=5e-4, total_steps=n_epochs, pct_start=0.15  # Lower max_lr
    )

    solar_train, demand_train, price_buy_train, price_sell_train = train_data
    n_train = solar_train.shape[0]
    n_batches = (n_train + batch_size - 1) // batch_size

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0

        # UPDATE CURRICULUM at the start of each epoch
        deg_weight, temp = model.update_curriculum(epoch, n_epochs)

        perm = torch.randperm(n_train)

        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, n_train)
            batch_idx = perm[start_idx:end_idx]
            s, d, pb, ps = [train_data[j][batch_idx] for j in range(4)]

            # Data augmentation with noise
            noise_scale = 0.05
            s = s + torch.randn_like(s) * noise_scale * s.mean(dim=1, keepdim=True)
            d = d + torch.randn_like(d) * noise_scale * d.mean(dim=1, keepdim=True)
            pb = pb + torch.randn_like(pb) * noise_scale * pb.mean(dim=1, keepdim=True)
            ps = ps + torch.randn_like(ps) * noise_scale * ps.mean(dim=1, keepdim=True)
            s, d, pb, ps = [torch.clamp(x, 0) for x in [s, d, pb, ps]]

            loss = model.compute_loss(s, d, pb, ps)

            if torch.isnan(loss):
                print(f"⚠️ NaN loss detected at epoch {epoch}, batch {i}")
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()

        scheduler.step()

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {epoch_loss/n_batches:.4f}, "
                  f"Deg_Weight: {deg_weight:.2f}, Temp: {temp:.3f}")

In [ ]:
class L2O_PWL(nn.Module):
    def __init__(self, params: BatteryParams, hidden_dim=1024):
        super().__init__()
        self.params = params
        self.T = params.T
        self.J = params.J
        self.temperature = params.temperature_init

        input_dim = 4 * self.T

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.mode_head = nn.Linear(hidden_dim, self.T * 3)
        self.power_head = nn.Linear(hidden_dim, self.T * 2)
        self.segment_allocation_ch = nn.Linear(hidden_dim, self.T * self.J)
        self.segment_allocation_dis = nn.Linear(hidden_dim, self.T * self.J)

        with torch.no_grad():
            self.mode_head.bias.data = self.mode_head.bias.data.reshape(self.T, 3)
            self.mode_head.bias.data[:, 0] = 0.2
            self.mode_head.bias.data[:, 1] = 0.2
            self.mode_head.bias.data[:, 2] = -0.4
            self.mode_head.bias.data = self.mode_head.bias.data.reshape(-1)

    def forward(self, solar, demand, price_buy, price_sell):
        batch_size = solar.shape[0]

        price_spread = price_buy - price_sell

        solar_max = 15.0  
        demand_max = 35.0 
        
        inputs = torch.cat([
            solar / solar_max,  # Fixed: covers test range
            demand / demand_max,
            price_buy / (price_buy.max(dim=1, keepdim=True)[0] + 1e-6),
            price_spread / (price_spread.max(dim=1, keepdim=True)[0] + 1e-6)
        ], dim=1)

        hidden = self.encoder(inputs)
        
        mode_logits = self.mode_head(hidden).reshape(batch_size, self.T, 3)
        if self.training:
            mode = F.gumbel_softmax(mode_logits, tau=self.temperature, hard=True)
        else:
            mode = F.one_hot(mode_logits.argmax(dim=-1), num_classes=3).float()

        powers = self.power_head(hidden).reshape(batch_size, self.T, 2)
        power_charge_raw = torch.sigmoid(powers[..., 0]) * self.params.max_charge_power
        power_discharge_raw = torch.sigmoid(powers[..., 1]) * self.params.max_discharge_power

        power_charge_total = mode[..., 0] * power_charge_raw
        power_discharge_total = mode[..., 1] * power_discharge_raw

        alloc_ch_logits = self.segment_allocation_ch(hidden).reshape(batch_size, self.T, self.J)
        alloc_dis_logits = self.segment_allocation_dis(hidden).reshape(batch_size, self.T, self.J)
        
        alloc_ch = F.softmax(alloc_ch_logits, dim=-1)
        alloc_dis = F.softmax(alloc_dis_logits, dim=-1)
        
        ch = power_charge_total.unsqueeze(-1) * alloc_ch
        dis = power_discharge_total.unsqueeze(-1) * alloc_dis

        soc_segments = self.compute_soc_segments(ch, dis)
        
        net_power = demand - solar + power_charge_total - power_discharge_total

        return {
            'ch': ch,
            'dis': dis,
            'power_charge': power_charge_total,
            'power_discharge': power_discharge_total,
            'mode': mode,
            'soc_segments': soc_segments,
            'soc_total': soc_segments.sum(dim=-1),  # Absolute SOC in kWh
            'net_power': net_power
        }

    def compute_soc_segments(self, ch, dis):

        batch_size = ch.shape[0]
        device = ch.device
        dt = self.params.delta_t
        
        # Segment max based on FULL capacity
        Ecap = self.params.battery_capacity
        soc_seg_max = Ecap / self.J  
        
        # Initialize segments representing energy from 0% up to initial_soc
        soc0_total = self.params.initial_soc * Ecap 
        
        soc_init = torch.zeros(batch_size, self.J, device=device)
        remaining = soc0_total
        for j in range(self.J):
            amount = min(soc_seg_max, remaining)
            soc_init[:, j] = amount
            remaining -= amount
        
        soc_trajectory = [soc_init]
        
        for t in range(self.T):
            soc_change = (ch[:, t, :] * self.params.charge_efficiency - 
                         dis[:, t, :] / self.params.discharge_efficiency) * dt
            soc_new = soc_trajectory[-1] + soc_change
            soc_trajectory.append(soc_new)
        
        return torch.stack(soc_trajectory, dim=1)

    def compute_loss(self, solar, demand, price_buy, price_sell):
        output = self.forward(solar, demand, price_buy, price_sell)
        dt = self.params.delta_t
        
        # Economic costs
        net_power = output['net_power']
        import_power = F.relu(net_power)
        export_power = F.relu(-net_power)
        import_cost = (import_power * price_buy * dt).sum(dim=1)
        export_revenue = (export_power * price_sell * dt).sum(dim=1)
        economic_cost = import_cost - export_revenue
        
        # PWL degradation cost 
        C_dis = torch.tensor(self.params.get_xu_degradation_costs(), 
                           device=output['dis'].device, dtype=output['dis'].dtype)
        degradation_cost = (output['dis'] * C_dis.view(1, 1, self.J) * dt).sum(dim=(1, 2))
        
        #  Per-segment SOC constraints (0 to soc_seg_max)
        Ecap = self.params.battery_capacity
        soc_seg_max = Ecap / self.J
        
        lower_viol_seg = F.relu(0 - output['soc_segments'])
        upper_viol_seg = F.relu(output['soc_segments'] - soc_seg_max)
        
        # ✅ FIXED: Total SOC constraints (10%-90% of full capacity)
        soc_total_abs = output['soc_total']  # Absolute SOC in kWh
        soc_min_abs = self.params.soc_min * Ecap  # 1 kWh
        soc_max_abs = self.params.soc_max * Ecap  # 9 kWh
        
        lower_viol_total = F.relu(soc_min_abs - soc_total_abs)
        upper_viol_total = F.relu(soc_total_abs - soc_max_abs)
        
        # Combined SOC violations
        soc_violations = (
            lower_viol_seg.sum(dim=(1, 2)) + 
            upper_viol_seg.sum(dim=(1, 2)) + 
            lower_viol_total.sum(dim=1) + 
            upper_viol_total.sum(dim=1)
        ) / (self.T * self.J)
        
        # Terminal constraint (absolute SOC)
        final_soc_total_abs = output['soc_total'][:, -1]
        target_soc_abs = self.params.final_soc * Ecap
        
        terminal_penalty = F.smooth_l1_loss(
            final_soc_total_abs, 
            torch.full_like(final_soc_total_abs, target_soc_abs), 
            reduction='none'
        )
        
        # Power tracking consistency
        computed_net = demand - solar + output['power_charge'] - output['power_discharge']
        track_residual = (net_power - computed_net).pow(2).mean(dim=1)
        
        # Total loss
        total_loss = (
            economic_cost.mean() +
            degradation_cost.mean() +
            self.params.soc_penalty_weight * soc_violations.mean() +
            self.params.terminal_penalty_weight * terminal_penalty.mean() +
            self.params.track_penalty_weight * track_residual.mean()
        )
        
        return total_loss

    def update_temperature(self, epoch, total_epochs):
        progress = epoch / total_epochs
        self.temperature = (
            self.params.temperature_init * (1 - progress) + 
            self.params.temperature_min * progress
        )
        return self.temperature


def train_l2o_pwl_model(model, train_data, n_epochs=1500, batch_size=48):
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=5e-4, total_steps=n_epochs, pct_start=0.15
    )

    solar_train, demand_train, price_buy_train, price_sell_train = train_data
    n_train = solar_train.shape[0]
    n_batches = (n_train + batch_size - 1) // batch_size

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0

        temp = model.update_temperature(epoch, n_epochs)

        perm = torch.randperm(n_train)

        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, n_train)
            batch_idx = perm[start_idx:end_idx]
            s, d, pb, ps = [train_data[j][batch_idx] for j in range(4)]

            # Data augmentation
            noise_scale = 0.05
            s = s + torch.randn_like(s) * noise_scale * s.mean(dim=1, keepdim=True)
            d = d + torch.randn_like(d) * noise_scale * d.mean(dim=1, keepdim=True)
            pb = pb + torch.randn_like(pb) * noise_scale * pb.mean(dim=1, keepdim=True)
            ps = ps + torch.randn_like(ps) * noise_scale * ps.mean(dim=1, keepdim=True)
            s, d, pb, ps = [torch.clamp(x, 0) for x in [s, d, pb, ps]]

            loss = model.compute_loss(s, d, pb, ps)

            if torch.isnan(loss):
                print(f"⚠️ NaN loss detected at epoch {epoch}, batch {i}")
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()

        scheduler.step()

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {epoch_loss/n_batches:.4f}, "
                  f"Temp: {temp:.3f}")

    return model

In [ ]:
def project_to_feasible(power_charge_nn, power_discharge_nn, params, 
                       check_feasibility_first=True):

    batch_size = power_charge_nn.shape[0]
    device = power_charge_nn.device
    T = power_charge_nn.shape[1]
    
    # Convert to numpy
    c_nn = power_charge_nn.detach().cpu().numpy()
    d_nn = power_discharge_nn.detach().cpu().numpy()
    
    # Constants
    dt = params.delta_t
    eta_c = params.charge_efficiency
    eta_d = params.discharge_efficiency
    soc_init = params.initial_soc * params.battery_capacity
    soc_min = params.soc_min * params.battery_capacity
    soc_max = params.soc_max * params.battery_capacity
    c_max = params.max_charge_power
    d_max = params.max_discharge_power
    
    # Pre-allocate results
    c_proj = c_nn.copy()
    d_proj = d_nn.copy()
    
    # Identify which samples need projection
    if check_feasibility_first:
        needs_projection = _check_soc_violations(c_nn, d_nn, params)
    else:
        needs_projection = np.ones(batch_size, dtype=bool)
    
    n_projected = needs_projection.sum()
    projection_distances = np.zeros(batch_size)
    
    if n_projected == 0:
        # All feasible, return as-is
        return (power_charge_nn, power_discharge_nn, 
                {'n_projected': 0, 'projection_distances': projection_distances})
    
    # Setup QP
    c = cp.Variable(T)
    d = cp.Variable(T)
    soc = cp.Variable(T + 1)
    
    c_nn_param = cp.Parameter(T)
    d_nn_param = cp.Parameter(T)
    
    # Objective: minimize L2 distance
    objective = cp.Minimize(
        cp.sum_squares(c - c_nn_param) + cp.sum_squares(d - d_nn_param)
    )
    
    # Constraints
    constraints = [soc[0] == soc_init]
    
    # SOC dynamics
    for t in range(T):
        constraints.append(
            soc[t+1] == soc[t] + (c[t] * eta_c - d[t] / eta_d) * dt
        )
    
    # Bounds
    constraints.extend([
        soc >= soc_min,
        soc <= soc_max,
        c >= 0,
        c <= c_max,
        d >= 0,
        d <= d_max,
    ])
    
    problem = cp.Problem(objective, constraints)
    
    # Solve for each sample that needs projection
    for b in range(batch_size):
        if not needs_projection[b]:
            continue
        
        # Update parameters
        c_nn_param.value = c_nn[b]
        d_nn_param.value = d_nn[b]
        
        try:
            problem.solve(
                solver=cp.OSQP,
                warm_start=True,
                verbose=False,
                eps_abs=1e-4,
                eps_rel=1e-4,
                max_iter=4000
            )
            
            if problem.status in ['optimal', 'optimal_inaccurate']:
                c_proj[b] = c.value
                d_proj[b] = d.value
                # Compute projection distance
                projection_distances[b] = np.sqrt(
                    np.sum((c_proj[b] - c_nn[b])**2) + 
                    np.sum((d_proj[b] - d_nn[b])**2)
                )
            else:
                print(f"⚠️ QP status '{problem.status}' for sample {b}")
        except Exception as e:
            print(f"⚠️ QP solve failed for sample {b}: {e}")
    
    # Convert back to torch
    projection_info = {
        'n_projected': n_projected,
        'projection_distances': projection_distances,
        'projected_indices': np.where(needs_projection)[0]
    }
    
    return (torch.from_numpy(c_proj).float().to(device),
            torch.from_numpy(d_proj).float().to(device),
            projection_info)


def _check_soc_violations(c_nn, d_nn, params):
    batch_size = c_nn.shape[0]
    T = c_nn.shape[1]
    needs_projection = np.zeros(batch_size, dtype=bool)
    
    dt = params.delta_t
    eta_c = params.charge_efficiency
    eta_d = params.discharge_efficiency
    soc_init = params.initial_soc * params.battery_capacity
    soc_min = params.soc_min * params.battery_capacity
    soc_max = params.soc_max * params.battery_capacity
    
    for b in range(batch_size):
        soc = soc_init
        violated = False
        
        for t in range(T):
            soc_change = (c_nn[b, t] * eta_c - d_nn[b, t] / eta_d) * dt
            soc = soc + soc_change
            
            if soc < soc_min - 1e-4 or soc > soc_max + 1e-4:
                violated = True
                break
        
        needs_projection[b] = violated
    
    return needs_projection